In [ ]:
# install dependency's
!pip install pandas openpyxl spacy transformers torch flair tokenizers
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 20.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of transformer-smaller-training-vocab to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving Job Descriptions.xlsx to Job Descriptions.xlsx


In [ ]:
df = pd.read_excel("Job Descriptions.xlsx")


In [ ]:
import os
print(os.listdir())


['.config', 'Job Descriptions.xlsx', 'sample_data']


In [ ]:
# Load the job descriptions
import pandas as pd
import spacy
from spacy.matcher import Matcher
from transformers import pipeline, AutoTokenizer, AutoModelForQuestionAnswering
from flair.models import TARSClassifier
from flair.data import Sentence
import torch

file_path = "Job Descriptions.xlsx"
df = pd.read_excel(file_path)
df.columns = ["Job_Description"] + df.columns.tolist()[1:]


In [ ]:
# 1. spaCy Pattern Matcher
nlp = spacy.load("en_core_web_sm")
matcher = Matcher(nlp.vocab)
patterns = [
    [{"LIKE_NUM": True}, {"LOWER": {"IN": ["years", "year"]}}],
    [{"LOWER": {"IN": ["minimum", "at", "least"]}}, {"LIKE_NUM": True}, {"LOWER": {"IN": ["years", "year"]}}],
    [{"LIKE_NUM": True}, {"LOWER": "to"}, {"LIKE_NUM": True}, {"LOWER": {"IN": ["years", "year"]}}],
]
matcher.add("ExperiencePatterns", patterns)

def extract_experience_spacy(text):
    doc = nlp(str(text))
    matches = matcher(doc)
    experiences = []
    for _, start, end in matches:
        span = doc[start:end]
        nums = [token.text for token in span if token.like_num]
        try:
            if len(nums) == 1:
                experiences.append(float(nums[0]))
            elif len(nums) == 2:
                avg = (float(nums[0]) + float(nums[1])) / 2
                experiences.append(avg)
        except ValueError:
            continue
    return max(experiences) if experiences else None


In [ ]:
# 2. BERT QA
bert_qa = pipeline("question-answering", model="bert-large-uncased-whole-word-masking-finetuned-squad")

def extract_experience_bert(text):
    try:
        result = bert_qa(question="What is the total experience required?", context=str(text))
        answer = result['answer']
        for word in answer.split():
            if word.replace('.', '', 1).isdigit():
                return float(word)
    except:
        return None
    return None


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-large-uncased-whole-word-masking-finetuned-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
# 3. RoBERTa QA
roberta_qa = pipeline("question-answering", model="deepset/roberta-base-squad2")

def extract_experience_roberta(text):
    try:
        result = roberta_qa(question="What is the total experience required?", context=str(text))
        answer = result['answer']
        for word in answer.split():
            if word.replace('.', '', 1).isdigit():
                return float(word)
    except:
        return None
    return None


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
# 4. TARS Zero-shot Classifier
tars = TARSClassifier.load("tars-base")
tars.add_and_switch_to_new_task(
    task_name="experience_estimation",
    label_dictionary=["0-2 years", "3-5 years", "6-10 years", "10+ years"],
    label_type="classification"
)

def extract_experience_tars(text):
    sentence = Sentence(str(text))
    try:
        tars.predict_zero_shot(sentence, ["0-2 years", "3-5 years", "6-10 years", "10+ years"])
        label = sentence.labels[0].value
        return {"0-2 years": 1.0, "3-5 years": 4.0, "6-10 years": 8.0, "10+ years": 12.0}.get(label, None)
    except:
        return None


2025-04-11 06:00:50,631 TARS initialized without a task. You need to call .add_and_switch_to_new_task() before training this model


In [ ]:
# 5. DistilBERT NER
from transformers import AutoModelForTokenClassification, TokenClassificationPipeline
ner_model = AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER")
ner_tokenizer = AutoTokenizer.from_pretrained("dslim/bert-base-NER")
ner_pipeline = pipeline("ner", model=ner_model, tokenizer=ner_tokenizer, aggregation_strategy="simple")

def extract_experience_ner(text):
    try:
        ents = ner_pipeline(str(text))
        for ent in ents:
            if "year" in ent['word'].lower() and any(char.isdigit() for char in ent['word']):
                digits = ''.join([c for c in ent['word'] if c.isdigit() or c == '.'])
                if digits: return float(digits)
    except:
        return None
    return None


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
# 6. GPT-style Prompting (simulated logic)
def extract_experience_prompt(text):
    for line in str(text).split(". "):
        if "experience" in line.lower():
            for word in line.split():
                if word.replace('.', '', 1).isdigit():
                    return float(word)
    return None


In [ ]:
# Apply all models
df["spaCy_Experience"] = df["Job_Description"].apply(extract_experience_spacy)
df["BERT_Experience"] = df["Job_Description"].apply(extract_experience_bert)
df["RoBERTa_Experience"] = df["Job_Description"].apply(extract_experience_roberta)
df["TARS_Experience"] = df["Job_Description"].apply(extract_experience_tars)
df["NER_Experience"] = df["Job_Description"].apply(extract_experience_ner)
df["Prompt_Experience"] = df["Job_Description"].apply(extract_experience_prompt)


In [ ]:
import os
os.listdir()


['.config', 'Job Descriptions.xlsx', 'sample_data']

In [ ]:
output_path = "JD_Experience_6Models.xlsx"
df.to_excel(output_path, index=False)


In [ ]:
from google.colab import files
files.download("JD_Experience_6Models.xlsx")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>